# PARC2026 — π0.5 Group-aware Dataset Cheap Ablation V3

`49_stage_training_dataset_from_drive.ipynb` のlocal stagingをこのNotebookへ統合しました。

CPU runtimeで `48_prefetch_training_dataset_to_drive.ipynb` を `DRIVE PREFETCH GATE: PASS` まで完了しておけば、
A100 runtimeでは **50を先頭から実行するだけ**で、

Drive → `/content` staging → group-aware manifest → leakage Gate → GA=8 runtime Gate

まで進みます。

`RUN_ABLATIONS=False` が初期値です。全Gateを確認してからTrueへ変更します。


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys, time

print('python:', sys.version)
print('platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

ROOT=Path('/content/parc2026')
for p in [ROOT, ROOT/'datasets', ROOT/'cache', ROOT/'outputs', ROOT/'vendor']:
    p.mkdir(parents=True, exist_ok=True)

REPO=ROOT/'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)

print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## Dataset staging（旧49を統合）

優先順位は explicit `PI05_DATASET_ROOT` → organizer combined → local staged public proxy → Drive prefetch です。

public proxyがlocalにもDriveにも無い場合は停止します。
**A100上でHugging Faceから約16GBの動画を再downloadしません。**


In [ ]:
PUBLIC_ROOT=ROOT/'datasets'/'public_libero_plus_v3_train'
ORGANIZER_ROOT=ROOT/'datasets'/'libero_combined_20hz'
DRIVE_ROOT=Path(os.environ.get(
    'PARC_DRIVE_DATASET_ROOT',
    '/content/drive/MyDrive/parc2026-cache/datasets/lerobot_libero_plus_v3_train'
))
MARKER='.parc_prefetch_complete.json'

def basic_ready(p):
    return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet')) and (p/'videos').exists()

def validate(root, manifest):
    return [
        row['path'] for row in manifest['files']
        if not (root/row['path']).exists()
        or (root/row['path']).stat().st_size != int(row['size_bytes'])
    ]

configured=os.environ.get('PI05_DATASET_ROOT')
DATASET_REVISION=None

if configured and basic_ready(Path(configured)):
    DATASET_ROOT=Path(configured)
    DATASET_ID=os.environ.get('PI05_DATASET_REPO_ID','local/libero_combined_20hz')
    DATASET_SOURCE='configured'
elif basic_ready(ORGANIZER_ROOT):
    DATASET_ROOT=ORGANIZER_ROOT
    DATASET_ID='local/libero_combined_20hz'
    DATASET_SOURCE='organizer_combined'
else:
    local_marker=PUBLIC_ROOT/MARKER
    manifest=json.loads(local_marker.read_text()) if local_marker.exists() else None
    local_ok=manifest is not None and not validate(PUBLIC_ROOT, manifest)

    if not local_ok:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')

        drive_marker=DRIVE_ROOT/MARKER
        if not drive_marker.exists():
            raise FileNotFoundError(
                f'{drive_marker} がありません。A100上では大容量downloadしません。'
                'CPU runtimeで 48_prefetch_training_dataset_to_drive.ipynb を先に実行してください。'
            )

        manifest=json.loads(drive_marker.read_text())
        bad=validate(DRIVE_ROOT, manifest)
        if bad:
            raise RuntimeError(f'Drive prefetch validation failed: {bad[:10]} total={len(bad)}')

        print('DRIVE SOURCE GATE: PASS')
        PUBLIC_ROOT.mkdir(parents=True, exist_ok=True)
        start=time.time()
        copied=skipped=0

        # 旧49の.cache/huggingfaceは不要。completion manifestのdataset本体だけstageする。
        for i,row in enumerate(manifest['files'],1):
            rel=Path(row['path'])
            src=DRIVE_ROOT/rel
            dst=PUBLIC_ROOT/rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            expected=int(row['size_bytes'])
            if dst.exists() and dst.stat().st_size == expected:
                skipped += 1
            else:
                shutil.copy2(src,dst)
                copied += 1
            if i==1 or i%10==0 or i==len(manifest['files']):
                print(f'stage {i}/{len(manifest["files"])} copied={copied} skipped={skipped}: {rel}')

        shutil.copy2(drive_marker, local_marker)
        print('stage wall sec:', round(time.time()-start,1))

    bad=validate(PUBLIC_ROOT, manifest)
    if bad:
        raise RuntimeError(f'local staging validation failed: {bad[:10]} total={len(bad)}')

    print('LOCAL STAGE GATE: PASS')
    DATASET_ROOT=PUBLIC_ROOT
    DATASET_ID=str(manifest['dataset_id'])
    DATASET_REVISION=str(manifest['revision'])
    DATASET_SOURCE='drive_prefetch_local_stage'

print('dataset source:', DATASET_SOURCE)
print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)


## HF token / safety setting

PaliGemma access用tokenは表示しません。
このセルの `RUN_ABLATIONS=False` は全Gate確認後だけTrueへ変更してください。


In [ ]:
from getpass import getpass

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok=userdata.get('HF_TOKEN')
    except Exception:
        tok=None
    os.environ['HF_TOKEN']=tok or getpass('HF token (PaliGemma access): ')

RUN_ABLATIONS=False
print('HF_TOKEN: set (not displayed)')
print('RUN_ABLATIONS:', RUN_ABLATIONS)


## Group-aware / Leakage / GA Gate + Cheap Screening

このセルはrepository側のrunnerを呼びます。

`RUN_ABLATIONS=False` の場合でも、
group-aware manifest・trajectory leakage・training manifest・GA=8 runtimeの全Gateまでは実行します。


In [ ]:
runner_env=os.environ.copy()
runner_env.update({
    'PARC_ROOT':str(ROOT),
    'PY_AI_REPO':str(REPO),
    'PI05_DATASET_ROOT':str(DATASET_ROOT),
    'PI05_DATASET_REPO_ID':DATASET_ID,
    'PI05_DATASET_REVISION':DATASET_REVISION or '',
    'HF_TOKEN':os.environ['HF_TOKEN'],
    'RUN_ABLATIONS':'true' if RUN_ABLATIONS else 'false',
})

subprocess.run(
    [sys.executable, str(REPO/'tools/colab/run_pi05_group_aware_ablation.py')],
    env=runner_env,
    check=True
)


## Exit criteria

`RUN_ABLATIONS=False` でまず以下を確認します。

- `LOCAL STAGE GATE: PASS`（public proxyの場合）
- `Group-aware Manifest Gate: PASS`
- `Trajectory Leakage Gate: PASS`
- `TRAINING MANIFEST SAFETY GATE: PASS`
- `GA Gate: PASS`

全部PASSしたら `RUN_ABLATIONS=True` に変更して最後の実行セルだけ再実行します。
